# 01 · ICNALE GRA — build the pool

*Holistic essay score band (Low / Mid / High)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** Asian-learner L2 English essays, each rated on holistic and analytic scales by many trained raters. This is an **automated writing evaluation** task: whole essays, not sentences.

**Difficulty of the labeling judgment:** ★★☆ — moderate, but a different shape of task: long texts and an ordered scale.

**Licence:** ⚠️ **Research use only — NOT redistributable.** Requires registration. Nothing derived from it may be committed to git or included in your submission bundle.  
**Cite:** Ishikawa, S. *The ICNALE Global Rating Archives.*

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, MEMBERS,
                    LABELS_ORDER, ROOT, OUT_DIR, POOL_PATH, DEMO_POOL_PATH,
                    SAMPLE_PATH, GOLD_PATH, PRED_PATH, ROUNDS_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

describe()                  # what this notebook is working on


## Step 1 — Get the data (this one is manual)

ICNALE GRA is released for research use behind a registration form that emails you a password. There is nothing to automate, and that is deliberate — the licence does not permit redistribution.

1. Register at <https://language.sakura.ne.jp/icnale/download.html> and wait for the password.
2. Download and unpack `ICNALE_GRA_2.x.zip`.
3. From its rating tables, export a CSV with **exactly two columns**, `text` and `score`.

In Colab, the cell below opens a file picker. Every other track downloads its corpus in one command and so keeps the raw data in the runtime — this one you cannot re-fetch without going back through the registration form, so it is worth keeping the file in your group's Drive folder and uploading it only once. The second option below does that.

⚠️ `data/raw/` is excluded from git and from your submission bundle, and anything with `icnale` in the name is excluded twice over. Leave it that way — the licence does not permit redistribution.

In [ ]:
# In Colab: uncomment to upload your essays_scores.csv
# from google.colab import files; files.upload()

RAW_FILE = "essays_scores.csv"        # the copy you just uploaded

# Or, having put it in your group's Drive folder once, use it from there and
# skip the upload every session:
# RAW_FILE = str(ROOT / "data" / "raw" / "icnale" / "essays_scores.csv")

# Say so here rather than three cells down, where the same problem arrives as
# a bare FileNotFoundError from inside `open`.
import os
if not os.path.isfile(RAW_FILE):
    raise FileNotFoundError(
        RAW_FILE + " is not here.\n"
        "This is the one track with no automatic download: register at "
        "https://language.sakura.ne.jp/icnale/download.html, export a CSV "
        "with a text column and a score column, then either uncomment the "
        "upload line above and run this cell again, or put the file in "
        "data/raw/icnale/ in your group's Drive folder and use the second "
        "RAW_FILE line.")
print("using", RAW_FILE)

## Step 2 — Look at the raw format

The cell prints the **distribution** of the scores, not just a couple of rows. You need that before step 3: it is what tells you where cutting the scale leaves you with three usable classes rather than one big one and two nearly empty ones.

In [ ]:
import csv

scores = []
with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    for row in reader:
        try:
            scores.append(float(row["score"]))
        except (TypeError, ValueError):
            pass

scores.sort()
print(len(scores), "scores · min", scores[0], "· max", scores[-1])
for q in (10, 25, 33, 50, 67, 75, 90):
    print("   ", str(q) + "th percentile:", scores[int(len(scores) * q / 100)])

### Reading CSV with `csv.DictReader`

`csv.DictReader` reads a CSV using its header row, so each row arrives as a dict keyed by column name — `row["score"]` rather than `row[1]`. That is what the cell above used to collect the scores.

**The two columns that matter:**

* `text` — the essay
* `score` — the holistic score. **It arrives as a string**, even when it looks like a number, so it has to be converted before it can be compared to a boundary.

The reshaping function below uses exactly these:

1. `open(..., encoding="utf-8-sig")` — strips the byte-order mark this file ships with, which would otherwise be glued to the first column name.
2. `csv.DictReader(handle)` — iterate rows as `{column: value}` dicts.
3. `float(raw_score)` inside a `try` — a non-numeric cell is counted and skipped rather than crashing the run. The function prints how many it skipped; if that number is not small, look at the file before trusting the rest.
4. `score < low_below` / `score < mid_below` — the two cut-offs you are about to choose. Everything else in the function is fixed; this is the whole decision.

## Step 3 — Reshape into the canonical schema

One decision, and it is entirely yours: ✏️ **where do the band boundaries go?**

There is no right answer sitting in the data waiting to be found. Two honest ways to choose, and they disagree:

- **From the rubric** — if the scale you are using says what a Low essay is, use that. Your classes will come out uneven, possibly badly, but they mean something outside your own study.
- **From the distribution** — cut at the 33rd and 67th percentiles (printed above) and your classes come out balanced. Your F1 is then easier to read, and your bands mean nothing except "bottom third of this sample".

Pick one, say which in `PLAN.md`, and report the boundaries as numbers. A band definition that exists only as an unexplained `4.0` in a notebook is not a scheme.

⚠️ These labels are **ordered** (Low < Mid < High) but they are *not* alphabetical. List them under `labels_order:` in `config.yaml` — Low, then Mid, then High — or the weighted κ gets computed over `High < Low < Mid`, which means nothing.

In [ ]:
import csv

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

def reshape_icnale(csv_path, low_below=4.0, mid_below=7.0):
    """Band a numeric holistic score into Low / Mid / High.

    THE CUT-OFFS ARE PLACEHOLDERS. 4 and 7 are not from the ICNALE rubric - they are
    round numbers. Where you put the boundaries decides how hard the task is and how
    balanced the classes are, so set them from the rubric you are actually using and
    say what you chose in your report.
    """
    rows = []
    skipped = 0
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:

        ### Read the CSV a row at a time ###
        for record in csv.DictReader(handle):    # DictReader gives each row as {column: value}.

            ### Pull the essay and its score ###
            text = (record.get("text") or "").strip()
            raw_score = (record.get("score") or "").strip()   # Still a string at this point.
            if not text or not raw_score:        # Nothing to band -> skip.
                continue

            ### Turn the score into a number ###
            try:
                score = float(raw_score)
            except ValueError:
                skipped = skipped + 1      # a non-numeric cell: report it, do not crash
                continue

            ### Apply the two cut-offs ###
            if score < low_below:                # Everything below the first boundary.
                label = "Low"
            elif score < mid_below:              # Between the two boundaries.
                label = "Mid"
            else:                                # At or above the second boundary.
                label = "High"
            rows.append({"id": 0, "text": text, "label": label})
    if skipped:
        print("  note: skipped", skipped, "row(s) whose score cell was not a number.")
    return reid(rows)

def validate(items, allowed=None):
    """Check the canonical schema, and raise on the first problem found.

    Deliberately explicit rather than `assert`: assertions vanish under `python -O`,
    and a silently unvalidated dataset is exactly the kind of thing that surfaces as a
    baffling metric three days later.
    """
    seen_ids = set()
    for position, item in enumerate(items):
        missing = {"id", "text", "label"} - set(item)
        if missing:
            raise ValueError("item " + str(position) + " is missing "
                             + str(sorted(missing)) + ": " + repr(item))
        if item["id"] in seen_ids:
            raise ValueError("duplicate id " + str(item["id"]))
        seen_ids.add(item["id"])
        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError("empty text at id " + str(item["id"]))
        if not isinstance(item["label"], str) or not item["label"]:
            raise ValueError("empty label at id " + str(item["id"]))
        if allowed is not None and item["label"] not in allowed:
            raise ValueError("label " + repr(item["label"]) + " at id "
                             + str(item["id"]) + " is not in " + str(sorted(allowed)))

In [ ]:
# ✏️ Step 3a · Cut the scale ─────────────────────────────────────
# Goal      : turn a numeric score into three bands, and be able to defend where.
# Shape     : rows = reshape_icnale(RAW_FILE, low_below=..., mid_below=...)
#             the defaults (4.0 / 7.0) are ROUND NUMBERS, not a rubric —
#             using them unchanged is a choice you would have to defend too
# Produce   : rows (a list)      ← later cells use this name
# Note      : run it a couple of ways and look at step 4 each time. Seeing
#             the counts move as you shift a boundary is the point.
# Careful   : whatever you settle on goes in PLAN.md as two numbers and a
#             reason. Do not re-cut after seeing your F1.

# ✏️ your code here — fill in each ____

# Two numbers, from the rubric or from the percentiles printed above.
# A score below low_below is Low; below mid_below is Mid; the rest High.
rows = reshape_icnale(RAW_FILE, low_below=____, mid_below=____)

print(len(rows), "essays")


## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
print("fields per item:", list(rows[0]))

# Peek at the first three. `context`, where a track has one, is trimmed: it is
# the whole passage and would bury everything else in this output.
for item in rows[:3]:
    preview = dict(item)
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)

## Step 5 — Save it

⚠️ Keep this file **out of git** and **out of your submission bundle**. `.gitignore` and `scripts/make_submission.py` both exclude anything with `icnale` in the name — please leave that in place.

In [ ]:
# Save the pool into your group's Drive folder, under the exact name notebook
# 02 will look for. POOL_PATH comes from config.yaml, so the two cannot drift.
import json

if TRACK not in ['icnale']:
    raise RuntimeError(
        "config.yaml says  track: " + str(TRACK) + "  but this is the icnale "
        "notebook, so saving now would put icnale data into "
        + POOL_PATH.name + ", which belongs to another track.\n"
        "Open config.yaml, set  track: to one of icnale,"
        " save it, then re-run the SETUP cell at the top of this notebook.")


# Check the shape before writing. Everything downstream — the sampling, the
# annotation sheet, the scoring — assumes every item has an id, a text and a
# label, and a pool that breaks that assumption does not fail here: it fails
# in notebook 03, after two people have annotated forty items.
validate(rows)

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.yaml`.